# FT-07 — Anatomie mémoire d'un pas d'entraînement, et parallélismes

**Navigation** : [Index](README.md) | [<< 06 — LoRA vision-langage](FT-06-Vision-Language-LoRA-Python.ipynb)

Un modèle de 7 milliards de paramètres ne tient pas dans les 8 Go d'un GPU consumer, et ce n'est pas une
question de qualité de code : c'est de l'arithmétique. Ce carnet rend ce budget **calculable poste par
poste**, puis le mesure là où cette machine le permet, en séparant à chaque étape ce qui est **mesuré ici**
de ce qui est **rapporté** par la littérature.

**Source.** *The Ultra-Scale Playbook: Training LLMs on GPU Clusters* — Nouamane Tazi, Ferdinand Mom,
Haojun Zhao, Phuc Nguyen, Mohamed Mekkouri, Leandro Werra, Thomas Wolf (Hugging Face), février 2025.
En libre accès : <https://huggingface.co/spaces/nanotron/ultrascale-playbook>.
Copie archivée au gisement partagé :
`G:\Mon Drive\MyIA\IA\Bibliographie IA\MachineLearning\2025 - Tazi Mom Zhao Nguyen Mekkouri Werra Wolf - The Ultra-Scale Playbook - Training LLMs on GPU Clusters (Hugging Face).pdf`
— sha256 `274a19a2577ed220cd3a102b4469c44310e4a7c8e8f8ebc36842d907cb51e127`, 14 059 172 octets.
Tout ce que ce carnet attribue à l'Ultra-Scale Playbook de Hugging Face est lu dans cette copie.

**Périmètre, dit avant de commencer.** Cette machine a un seul GPU consumer, et le noyau canonique
`python3` porte `torch 2.13.0+cpu` : aucune mesure multi-GPU, aucun cluster. Les résultats du playbook qui
exigent 512 GPU restent **rapportés** ; la table d'honnêteté de la partie 7 sépare les mesures locales de
ce qui vient de la source, ligne par ligne.

**Pourquoi ce carnet ouvre la série FineTuning.** LoRA et QLoRA, mesurés dans FT-00d et FT-02, sont des
contournements d'un budget mémoire. Ce carnet rend ce budget explicite : ce que les carnets suivants
subissent, celui-ci le calcule et le mesure.

## 1. Combien d'octets par paramètre

Avant de parler de mémoire, il faut savoir de combien de paramètres on parle. Le playbook donne la
formule usuelle pour un Transformer décodeur, en négligeant les biais et les couches de normalisation :

$$\text{paramètres} \approx 12 \cdot l \cdot h^2 + V \cdot h$$

avec $l$ le nombre de couches, $h$ la dimension cachée et $V$ la taille du vocabulaire. Le $12$ vient de
l'attention ($4h^2$ pour $W_q, W_k, W_v, W_o$) plus le MLP au facteur 4 ($8h^2$).

On construit un modèle jouet qui suit exactement cette architecture, avec les embeddings liés (*tied*,
comme dans la formule : la tête de sortie réutilise la matrice d'embedding, elle ne coûte rien de plus).

In [1]:
import time

import torch
import torch.utils.checkpoint as checkpoint
from torch import nn

torch.manual_seed(0)


def octets(t):
    """Octets logiques d'un tenseur.

    `t.untyped_storage().nbytes()` serait un mauvais instrument ici : il compte le
    stockage ENTIER qui porte le tenseur. Des qu'un tenseur est une vue (un `chunk`,
    une `transpose`, un `view`), plusieurs tenseurs partagent le meme stockage et le
    compteur facture la meme allocation autant de fois qu'il y a de vues.
    """
    return t.numel() * t.element_size()


class BlocTransformer(nn.Module):
    """Un bloc pre-norm : attention multi-tetes puis MLP, avec residus."""

    def __init__(self, h, tetes, facteur_mlp=4):
        super().__init__()
        self.norme1 = nn.LayerNorm(h)
        self.qkv = nn.Linear(h, 3 * h, bias=False)
        self.proj = nn.Linear(h, h, bias=False)
        self.norme2 = nn.LayerNorm(h)
        self.mlp = nn.Sequential(
            nn.Linear(h, facteur_mlp * h, bias=False),
            nn.GELU(),
            nn.Linear(facteur_mlp * h, h, bias=False),
        )
        self.tetes = tetes

    def forward(self, x):
        b, s, h = x.shape
        y = self.norme1(x)
        q, k, v = self.qkv(y).chunk(3, dim=-1)
        q, k, v = (t.view(b, s, self.tetes, h // self.tetes).transpose(1, 2) for t in (q, k, v))
        a = torch.nn.functional.scaled_dot_product_attention(q, k, v)
        a = a.transpose(1, 2).reshape(b, s, h)
        x = x + self.proj(a)
        return x + self.mlp(self.norme2(x))


class PetitGPT(nn.Module):
    """Decodeur minimal dont le compte de parametres suit la formule du playbook."""

    def __init__(self, vocab, h, couches, tetes):
        super().__init__()
        self.embed = nn.Embedding(vocab, h)
        self.blocs = nn.ModuleList(BlocTransformer(h, tetes) for _ in range(couches))
        self.norme = nn.LayerNorm(h)
        self.tete = nn.Linear(h, vocab, bias=False)
        self.tete.weight = self.embed.weight  # embeddings lies : la tete ne coute rien de plus

    def forward(self, ids):
        x = self.embed(ids)
        for bloc in self.blocs:
            x = bloc(x)
        return self.tete(self.norme(x))


VOCAB, H, COUCHES, TETES = 4096, 256, 4, 8
modele = PetitGPT(VOCAB, H, COUCHES, TETES)

reel = sum(p.numel() for p in modele.parameters())
analytique = 12 * COUCHES * H**2 + VOCAB * H
print(f"parametres comptes par PyTorch : {reel:>12,d}")
print(f"parametres selon 12*l*h^2 + V*h  : {analytique:>12,d}")
print(f"ecart relatif                    : {(reel - analytique) / analytique:>12.4%}")
print(f"poids du vocabulaire dans le total : {VOCAB * H / reel:>9.2%}")

parametres comptes par PyTorch :    4,198,912
parametres selon 12*l*h^2 + V*h  :    4,194,304
ecart relatif                    :      0.1099%
poids du vocabulaire dans le total :    24.97%


**Lecture.** La formule tombe à **0,11 %** du compte réel : l'écart, ce sont les deux `LayerNorm` par
bloc et la normale finale, que la formule ignore — négligeable à cette échelle, et la source l'assume.
PyTorch compte **4 198 912** paramètres contre **4 194 304** annoncés par la formule.

Le point qui compte pour la suite est le dernier chiffre : **25 % des paramètres sont dans la matrice
d'embedding**. Ce n'est pas un détail de ce modèle jouet, c'est une propriété générale des petits
vocabulaires à grande dimension — et c'est précisément le poste que les techniques de la partie 4
traitent différemment du reste, puisque ces poids ne reçoivent de gradient que sur les jetons présents
dans le lot.

## 2. Le budget d'un pas d'entraînement : les six postes

Le playbook énumère les postes mémoire d'un entraînement en précision mixte avec Adam. On peut les
matérialiser — allouer réellement chaque tenseur et compter ses octets — plutôt que de les additionner
de tête. Six postes, et leurs tailles se lisent directement dans le `dtype` :

| Poste | `dtype` | Octets/paramètre | Pourquoi il existe |
|---|---|---|---|
| Poids maîtres | fp32 | 4 | l'accumulateur qui reçoit les mises à jour de l'optimiseur |
| Poids de travail | bf16 | 2 | ce que le `forward` consomme |
| Gradients de travail | bf16 | 2 | ce que le `backward` produit |
| Gradients accumulés | fp32 | 4 | la sommation des micro-lots sans dérive |
| Adam $m$ | fp32 | 4 | moyenne glissante du gradient |
| Adam $v$ | fp32 | 4 | moyenne glissante du gradient au carré |

Le total fait **20 octets par paramètre** — c'est la version avec accumulation fp32 des gradients. Le
playbook chiffre ailleurs **16 octets** (sans ce poste) ; les deux sont justes, ils ne décrivent pas le
même réglage, et cette distinction revient en partie 4.

In [2]:
n = sum(p.numel() for p in modele.parameters())

postes = {
    "poids maitres (fp32)": torch.zeros(n, dtype=torch.float32),
    "poids de travail (bf16)": torch.zeros(n, dtype=torch.bfloat16),
    "gradients de travail (bf16)": torch.zeros(n, dtype=torch.bfloat16),
    "gradients accumules (fp32)": torch.zeros(n, dtype=torch.float32),
    "Adam m (fp32)": torch.zeros(n, dtype=torch.float32),
    "Adam v (fp32)": torch.zeros(n, dtype=torch.float32),
}

total = 0
for nom, t in postes.items():
    o = octets(t)
    total += o
    print(f"{nom:<28} {o:>12,d} o  {o / 1024**2:7.2f} Mio  {o / n:4.1f} o/param")

print("-" * 78)
print(f"{'TOTAL etat d entrainement':<28} {total:>12,d} o  {total / 1024**2:7.2f} Mio  {total / n:4.1f} o/param")

# Ce que couterait un modele 7B avec le meme ledger
for milliards in (1, 7, 70):
    print(f"  extrapolation {milliards:>2} Md de parametres : {total / n * milliards * 1e9 / 1024**3:8.1f} Gio d'etat")

poids maitres (fp32)           16,795,648 o    16.02 Mio   4.0 o/param
poids de travail (bf16)         8,397,824 o     8.01 Mio   2.0 o/param
gradients de travail (bf16)     8,397,824 o     8.01 Mio   2.0 o/param
gradients accumules (fp32)     16,795,648 o    16.02 Mio   4.0 o/param
Adam m (fp32)                  16,795,648 o    16.02 Mio   4.0 o/param
Adam v (fp32)                  16,795,648 o    16.02 Mio   4.0 o/param
------------------------------------------------------------------------------
TOTAL etat d entrainement      83,978,240 o    80.09 Mio  20.0 o/param
  extrapolation  1 Md de parametres :     18.6 Gio d'etat
  extrapolation  7 Md de parametres :    130.4 Gio d'etat
  extrapolation 70 Md de parametres :   1303.9 Gio d'etat


**Lecture.** Les six postes totalisent **83 978 240 octets**, soit **exactement 20,0 octets par
paramètre** — l'arithmétique des `dtype` est vérifiée par l'allocation réelle, pas par une formule
recopiée. Quatre des six postes sont en fp32 : **16 des 20 octets ne servent pas au calcul mais à
l'optimiseur et à son accumulation**.

L'extrapolation donne le chiffre qui décide de tout : **130 Gio pour 7 milliards de paramètres**. Un GPU
consumer de 8 Go en porte un seizième. Ce n'est pas un problème de réglage, c'est le point de départ de
toute la série FineTuning : à partir d'ici, chaque technique du carnet suivant paie ou économise sur
l'un de ces six postes, et rien d'autre.

## 3. Les activations : le poste qui dépend de la séquence

Les six postes ci-dessus sont **constants** — ils ne bougent pas quand on allonge la séquence ni quand on
augmente le lot. Les activations, elles, sont proportionnelles à `lot × séquence`, et ce sont elles qui
font passer un modèle qui « tenait » à l'`OutOfMemoryError` sur un lot deux fois plus long.

Le playbook donne une formule issue des travaux de recomputation de NVIDIA :

$$\text{octets} = l \cdot s \cdot b \cdot h \cdot \left(34 + 5 \cdot \frac{a \cdot s}{h}\right)$$

avec $l$ couches, $s$ la longueur de séquence, $b$ le lot, $h$ la dimension cachée et $a$ le nombre de
têtes. Le terme $5as/h$ est le terme d'attention — c'est le seul qui soit **quadratique** en $s$.

Mesurer plutôt que croire : on intercepte les tenseurs réellement conservés pour le `backward` avec
`torch.autograd.graph.saved_tensors_hooks`, et on compte leurs octets logiques.

In [3]:
def mesurer_activations(modele, lot, seq, avec_checkpoint=False):
    """Octets reellement conserves pour le backward, via saved_tensors_hooks."""
    total = [0]
    vus = [0]

    def pack(t):
        if isinstance(t, torch.Tensor):
            total[0] += octets(t)
            vus[0] += 1
        return t

    for p in modele.parameters():
        p.grad = None
    ids = torch.randint(0, VOCAB, (lot, seq))
    with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
        if avec_checkpoint:
            x = modele.embed(ids)
            for bloc in modele.blocs:
                x = checkpoint.checkpoint(bloc, x, use_reentrant=False)
            sortie = modele.tete(modele.norme(x))
        else:
            sortie = modele(ids)
        sortie.sum().backward()
    return total[0], vus[0]


def formule_activations(couches, seq, lot, h, tetes):
    """Formule du playbook, en octets bf16 (2 octets par valeur)."""
    return 2 * couches * seq * lot * h * (34 + 5 * tetes * seq / h)


modele_bf16 = PetitGPT(VOCAB, H, COUCHES, TETES).to(torch.bfloat16)
print(f"{'lot':>4} {'seq':>5} {'mesure':>13} {'formule':>13} {'ratio':>7}")
for lot, seq in [(1, 128), (1, 256), (1, 512), (2, 512)]:
    mesure, nb = mesurer_activations(modele_bf16, lot, seq)
    form = formule_activations(COUCHES, seq, lot, H, TETES)
    print(f"{lot:>4} {seq:>5} {mesure:>13,d} {int(form):>13,d} {mesure / form:>7.3f}")

 lot   seq        mesure       formule   ratio


   1   128    13,007,360    14,155,776   0.919


   1   256    17,616,896    38,797,312   0.454


   1   512    26,835,968   119,537,664   0.224


   2   512    45,274,112   239,075,328   0.189


**Lecture.** La formule est bonne à **128 jetons** (ratio $0{,}919$, soit 8 % d'erreur) et devient
franchement pessimiste à mesure que la séquence s'allonge : $0{,}454$ à 256, **$0{,}224$ à 512**. Autrement
dit, à 512 jetons l'implémentation réelle consomme **quatre fois et demie moins** que ce que la formule
prédit. Ce n'est pas la formule qui se trompe sur ce modèle : c'est elle qui décrit un autre code.

Le nombre de tenseurs conservés est d'ailleurs **identique dans les quatre cas** ($104$) : ce n'est pas le
nombre d'objets qui change, c'est leur taille — donc c'est bien le terme d'attention qui est surestimé.
Deux causes se cumulent, et la cellule suivante les sépare : la formule compte le terme $5as/h$ à
l'épaisseur du fp32 alors que ce modèle calcule en bf16, et `scaled_dot_product_attention` ne matérialise
jamais la matrice $s \times s$ que ce terme suppose.

In [4]:
class BlocManuel(BlocTransformer):
    """Meme bloc, mais l'attention est ecrite a la main : la matrice de scores est materielisee."""

    def forward(self, x):
        b, s, h = x.shape
        y = self.norme1(x)
        q, k, v = self.qkv(y).chunk(3, dim=-1)
        q, k, v = (t.view(b, s, self.tetes, h // self.tetes).transpose(1, 2) for t in (q, k, v))
        scores = (q @ k.transpose(-2, -1)) / (h // self.tetes) ** 0.5
        a = torch.softmax(scores, dim=-1) @ v
        a = a.transpose(1, 2).reshape(b, s, h)
        x = x + self.proj(a)
        return x + self.mlp(self.norme2(x))


class PetitGPTManuel(PetitGPT):
    """Le meme modele, bloc par bloc, en ecriture explicite."""

    def __init__(self, vocab, h, couches, tetes):
        super().__init__(vocab, h, couches, tetes)
        self.blocs = nn.ModuleList(BlocManuel(h, tetes) for _ in range(couches))


manuel = PetitGPTManuel(VOCAB, H, COUCHES, TETES).to(torch.bfloat16)
print(f"{'lot':>4} {'seq':>5} {'fusionnee':>13} {'explicite':>13} {'formule':>13} {'xSDPA':>7}")
for lot, seq in [(1, 128), (1, 256), (1, 512), (2, 512)]:
    m_fusion, _ = mesurer_activations(modele_bf16, lot, seq)
    m_expl, _ = mesurer_activations(manuel, lot, seq)
    form = formule_activations(COUCHES, seq, lot, H, TETES)
    print(f"{lot:>4} {seq:>5} {m_fusion:>13,d} {m_expl:>13,d} {int(form):>13,d} {m_expl / m_fusion:>7.2f}")

 lot   seq     fusionnee     explicite       formule   xSDPA


   1   128    13,007,360    14,825,984    14,155,776    1.14


   1   256    17,616,896    25,448,448    38,797,312    1.44


   1   512    26,835,968    59,276,288   119,537,664    2.21


   2   512    45,274,112   110,154,752   239,075,328    2.43


**Lecture.** L'ablation isole la première cause : avec la matrice de scores écrite explicitement, la
consommation passe de **26 835 968** à **59 276 288** octets à 512 jetons — exactement **2,21 fois** plus,
pour des calculs mathématiquement identiques. La formule se rapproche alors de la mesure ($0{,}496$ contre
$0{,}224$), mais reste deux fois trop haute : la seconde cause est bien là, et elle est *dans la formule* —
ses coefficients d'attention, $34$ et $5$, sont hérités d'un décompte fp32 (4 octets par valeur) alors que
le modèle calcule en bf16, et le résidu mesuré le montre : $59\,276\,288 \times 2 = 118\,552\,576$, à 1 %
de la formule $119\,537\,664$ — l'écart restant est ce facteur, pas une cause indépendante. Elle compte de
plus des tenseurs que PyTorch ne conserve pas.

Le résultat utile n'est pas « la formule est fausse » : c'est que **le coût des activations n'est pas une
propriété du modèle, c'est une propriété du couple (modèle, noyau d'attention)**. Deux implémentations
équivalentes au même résultat numérique diffèrent d'un facteur 2,2 en mémoire. Un tableau de dimensionnement
qui ne nomme pas son noyau d'attention ne prédit rien.

## 4. Recomputing : échanger de la mémoire contre du calcul

Le playbook distingue trois stratégies pour les activations :

- **tout garder** — le `backward` est gratuit, la mémoire ne l'est pas ;
- **recomputation sélective** — on ne conserve que les entrées de bloc, et on recalcule l'intérieur de
  chaque bloc au `backward`. C'est `torch.utils.checkpoint`, et la source annonce **+30 à 40 % de calcul**
  pour la version complète ;
- **recomputation intégrale** — on jette même les entrées.

La promesse est donc un **échange explicite** : moins de mémoire, plus de temps. Ce qui compte
pratiquement, c'est le taux de change. On le mesure au lieu de le supposer — avec échauffement et
minimum de trois passes, parce qu'un premier appel paie l'initialisation des noyaux et donnerait
un ratio de temps fantaisiste.

In [5]:
def temps_minimum(avec_checkpoint, passes=6, jetees=2):
    """Minimum de plusieurs passes, apres en avoir jete quelques-unes : le premier appel
    paie l'initialisation des noyaux, et un run unique mesurerait cet echauffement."""
    mesures = []
    for _ in range(passes):
        t0 = time.perf_counter()
        mesurer_activations(modele_bf16, 1, 512, avec_checkpoint)
        mesures.append(time.perf_counter() - t0)
    return min(mesures[jetees:])


t_sans = temps_minimum(False)
t_avec = temps_minimum(True)
memoire_sans, _ = mesurer_activations(modele_bf16, 1, 512)
memoire_avec, _ = mesurer_activations(modele_bf16, 1, 512, avec_checkpoint=True)

print(f"activations sans recomputation : {memoire_sans:>12,d} o  en {t_sans * 1000:7.1f} ms")
print(f"activations avec recomputation : {memoire_avec:>12,d} o  en {t_avec * 1000:7.1f} ms")
print(f"memoire divisee par             : {memoire_sans / memoire_avec:>12.2f}")
print(f"temps multiplie par             : {t_avec / t_sans:>12.2f}")

activations sans recomputation :   26,835,968 o  en  8869.1 ms
activations avec recomputation :    3,677,184 o  en  9100.3 ms
memoire divisee par             :         7.30
temps multiplie par             :         1.03


**Lecture.** La mémoire des activations tombe de **26 835 968** à **3 677 184** octets — divisée par
**7,30** — pour un surcoût de temps mesuré de **1,03**, soit **3 %**.

Ces deux chiffres n'ont pas le même statut, et il faut le dire :

- le **facteur mémoire** est une propriété de la structure du graphe. Il est déterministe, il ne dépend
  ni de la machine ni de la charge, et il se reproduira au bit près ;
- le **facteur de temps** est une mesure de plateforme. À 8,9 secondes par passe, ce poste exécute le
  bf16 sur un chemin lent ; une passe avant supplémentaire y pèse peu devant un `backward` déjà très
  coûteux, d'où 3 % là où le playbook annonce **+30 à 40 % sur GPU**. Ce n'est pas une contradiction de
  la source : c'est la démonstration que **le taux de change d'une recomputation ne se lit pas sur la
  mauvaise plateforme**. Le facteur mémoire, lui, se transpose.

C'est le point le plus utile de cette partie : les deux moitiés de l'échange n'ont pas la même
portabilité. Une économie de mémoire se calcule ; un coût de calcul se mesure, et seulement sur le
matériel où l'on compte l'exploiter.

## 5. ZeRO : partager l'état au lieu de le dupliquer

Les vingt octets de la partie 2 sont payés **sur chaque GPU**. Avec du parallélisme de données, quatre
cartes détiennent quatre copies identiques de ces vingt octets — et neuf dixièmes de cette mémoire est
identique d'un rang à l'autre. ZeRO consiste à découper ces postes en tranches et à n'en garder qu'une
par rang :

| Étage | Ce qui est partitionné | Ce qui reste entier sur chaque rang |
|---|---|---|
| **ZeRO-1** | les états de l'optimiseur ($m$, $v$) | poids, gradients |
| **ZeRO-2** | états de l'optimiseur **et** gradients | poids |
| **ZeRO-3** | états, gradients **et** poids | rien |

Plutôt que de recopier un tableau, on le **dérive** du ledger mesuré : on déclare quels postes chaque
étage partitionne, et le calcul fait le reste — chaque étage doit rendre le même total quand $ND = 1$,
ce qui est le contrôle de conservation.

```mermaid
flowchart TD
    E["Etat d'entrainement complet<br/>20 octets par parametre"] --> Z1["ZeRO-1<br/>optimiseur partitionne<br/>12 + 8/ND"]
    Z1 --> Z2["ZeRO-2<br/>gradients aussi partitionnes<br/>6 + 14/ND"]
    Z2 --> Z3["ZeRO-3<br/>poids aussi partitionnes<br/>20/ND"]
    Z3 --> C["Chaque rang ne detient plus<br/>qu'une tranche de chaque poste"]
```

Les trois étages sont **emboîtés** : ZeRO-2 partitionne tout ce que ZeRO-1 partitionne, plus les
gradients. C'est pourquoi les restes rétrécissent d'un étage au suivant au lieu de changer de nature.

In [6]:
partitionnes = {
    "ZeRO-1": {"Adam m (fp32)", "Adam v (fp32)"},
    "ZeRO-2": {"Adam m (fp32)", "Adam v (fp32)",
               "gradients de travail (bf16)", "gradients accumules (fp32)"},
    "ZeRO-3": set(postes),  # tout, y compris les poids
}


def octets_par_param(nom_etage, nd):
    """Octets residant sur UN rang, quand l'etage est reparti sur nd rangs."""
    decoupes = partitionnes[nom_etage]
    total = 0.0
    for nom, t in postes.items():
        o = octets(t) / n
        total += o / nd if nom in decoupes else o
    return total


print(f"{'etage':<8} " + "".join(f"{'ND=' + str(nd):>12}" for nd in (1, 2, 4, 8, 64, 512)))
for etage in ["ZeRO-0", "ZeRO-1", "ZeRO-2", "ZeRO-3"]:
    ligne = f"{etage:<8} "
    for nd in (1, 2, 4, 8, 64, 512):
        v = 20.0 if etage == "ZeRO-0" else octets_par_param(etage, nd)
        ligne += f"{v:>12.2f}"
    print(ligne)

print()
print("Controle de conservation : a ND=1, chaque etage doit rendre 20.0 o/param")
for etage in ["ZeRO-1", "ZeRO-2", "ZeRO-3"]:
    print(f"  {etage} a ND=1 -> {octets_par_param(etage, 1):.2f} o/param")

etage            ND=1        ND=2        ND=4        ND=8       ND=64      ND=512
ZeRO-0          20.00       20.00       20.00       20.00       20.00       20.00
ZeRO-1          20.00       16.00       14.00       13.00       12.12       12.02
ZeRO-2          20.00       13.00        9.50        7.75        6.22        6.03
ZeRO-3          20.00       10.00        5.00        2.50        0.31        0.04

Controle de conservation : a ND=1, chaque etage doit rendre 20.0 o/param
  ZeRO-1 a ND=1 -> 20.00 o/param
  ZeRO-2 a ND=1 -> 20.00 o/param
  ZeRO-3 a ND=1 -> 20.00 o/param


**Lecture.** Le contrôle de conservation passe : à $ND = 1$, les trois étages rendent bien **20,00
octets par paramètre**. Ce n'est pas une formalité — c'est ce qui distingue un tableau dérivé d'un tableau
recopié, et c'est ce qui a permis de voir que le ledger du playbook (16 octets, sans accumulation fp32 des
gradients) et le nôtre (20 octets) ne donnent pas les mêmes colonnes.

Trois choses se lisent dans la grille :

1. **ZeRO-3 à 8 rangs divise par 8** ($2{,}50$ o/param) : les 130 Gio du modèle 7B tombent à **16,3 Gio
   par carte**. C'est le seul étage qui rend un modèle trop gros pour une carte entraînable sur huit.
2. **Les rendements décroissent** : sur ZeRO-2, passer de 8 à 512 rangs ne fait gagner que 22 %
   ($7{,}75 \to 6{,}03$) alors que le volume de communication est multiplié par 64. ZeRO-1 tend vers
   **12** o/param à mesure que les rangs croissent ($16{,}00$ à 2 rangs, $13{,}00$ à 8, $12{,}12$ à 64) :
   ce qui n'est pas partitionné ne se partage pas, et l'asymptote est le poste non partitionné.
3. **Le plancher n'est pas zéro** : à 512 rangs, ZeRO-3 laisse 0,04 o/param d'état, mais les activations
   de la partie 3, elles, **ne sont pas partitionnées du tout** — et pour une longue séquence elles
   dominent. C'est pourquoi le playbook enchaîne sur le parallélisme de tenseur, puis sur le parallélisme
   de contexte, plutôt que de s'arrêter à ZeRO-3.

## 6. Ce que ZeRO coûte en communication

ZeRO ne fait pas que déplacer de la mémoire : chaque poste partitionné crée une opération collective. Le
`backward` se termine par un **all-reduce** des gradients sur tous les rangs, et c'est lui qui décide si
l'échange mémoire/bande passante est rentable.

L'algorithme de référence est l'**anneau** (*ring all-reduce*) : chaque rang envoie au suivant, en
$2(ND-1)$ étapes — un *reduce-scatter* puis un *all-gather* — de sorte que le volume échangé par un rang
soit $\frac{2(ND-1)}{ND} M$, où $M$ est la taille du tampon. Ce rapport **tend vers 2** quand $ND$ grandit :
doubler les cartes ne double pas le trafic, contrairement à une diffusion naïve en étoile où le serveur
devient le goulot.

On simule l'anneau sur de vrais tenseurs — pas sur des nombres de paquets, mais sur les données elles-mêmes,
pour vérifier **deux** choses à la fois : que le volume suit la formule, et que le résultat du reduce est
numériquement juste sur chaque rang.

In [7]:
def anneau_all_reduce(nd, taille):
    """Anneau all-reduce sur des tenseurs reels : reduce-scatter puis all-gather.

    Le rang i detient au depart `base * (i + 1)`. Apres reduction, chaque rang doit
    detenir `base * somme(1..nd)`. On compte les octets REELLEMENT echanges, somme
    sur tous les rangs, sans passer par une formule.
    """
    base = torch.arange(1, taille + 1, dtype=torch.float64)
    tampons = [base * (i + 1) for i in range(nd)]
    morceaux = [list(torch.chunk(t, nd)) for t in tampons]
    transferts = 0

    for etape in range(nd - 1):  # reduce-scatter
        suivants = [[morceaux[i][j].clone() for j in range(nd)] for i in range(nd)]
        for i in range(nd):
            dest = (i + 1) % nd
            idx = (i - etape) % nd
            suivants[dest][idx] = morceaux[dest][idx] + morceaux[i][idx]
            transferts += octets(morceaux[i][idx])
        morceaux = suivants

    for etape in range(nd - 1):  # all-gather
        suivants = [[morceaux[i][j].clone() for j in range(nd)] for i in range(nd)]
        for i in range(nd):
            dest = (i + 1) % nd
            idx = (i - etape + 1) % nd
            suivants[dest][idx] = morceaux[i][idx]
            transferts += octets(morceaux[i][idx])
        morceaux = suivants

    return transferts, [torch.cat(m) for m in morceaux]


print(f"{'ND':>3} {'tampon':>10} {'echange':>13} {'2(ND-1)M':>13} {'ratio':>7} {'juste':>6}")
for nd in (2, 4, 8):
    for taille in (1024, 65536):
        transferts, rangs = anneau_all_reduce(nd, taille)
        attendu = torch.arange(1, taille + 1, dtype=torch.float64) * sum(range(1, nd + 1))
        juste = all(torch.equal(r, attendu) for r in rangs)
        volume = 2 * (nd - 1) * taille * 8
        print(f"{nd:>3} {taille * 8:>10,d} {transferts:>13,d} {volume:>13,d} "
              f"{transferts / volume:>7.4f} {str(juste):>6}")

 ND     tampon       echange      2(ND-1)M   ratio  juste
  2      8,192        16,384        16,384  1.0000   True
  2    524,288     1,048,576     1,048,576  1.0000   True
  4      8,192        49,152        49,152  1.0000   True
  4    524,288     3,145,728     3,145,728  1.0000   True
  8      8,192       114,688       114,688  1.0000   True
  8    524,288     7,340,032     7,340,032  1.0000   True


**Lecture.** Le ratio vaut **1,0000** sur les six configurations, et le résultat est **numériquement juste
sur chaque rang** : l'anneau est optimal au sens de la bande passante, et il l'est sans exception pour
$ND \in \{2, 4, 8\}$ aussi bien sur un tampon de 8 Kio que de 512 Kio. Le volume total échangé est
exactement $2(ND-1)M$ — mais ce qui compte pour une carte, c'est sa part, $\frac{2(ND-1)}{ND}M$, qui tend
vers $2M$ sans jamais l'atteindre : à 8 rangs elle vaut $1{,}75M$, à 512 rangs $1{,}996M$. **Le coût par
carte ne double pas quand on double le nombre de cartes.**

L'arithmétique qui suit est une projection, pas une mesure. Pour le modèle 7B de la partie 2, l'all-reduce
porte les gradients bf16, soit 14 Go. À 8 rangs, chaque carte échange $1{,}75 \times 14$ = **24,5 Go par
pas**. Sur un interconnect de 100 Gbit/s (12,5 Go/s), cela fait **2 secondes par pas** de communication
pure ; sur du NVLink à 900 Go/s, 27 ms. Le même calcul explique la règle empirique que donne le playbook —
rester sous une cinquantaine de rangs par groupe de données, et recouvrir la communication par le calcul —
puisque le temps de communication cesse d'être négligeable dès qu'il approche le temps du `backward`.

## 7. Table d'honnêteté : ce que ce carnet mesure, et ce qu'il rapporte

Un carnet sur un article de 512 GPU, exécuté sur un poste sans GPU, doit dire lequel des deux parle à
chaque ligne. Le tableau ci-dessous est la règle de lecture de tout ce qui précède.

| Affirmation | Statut sur ce poste |
|---|---|
| 20 octets/paramètre pour l'état d'entraînement | **mesuré** — allocation réelle, six postes |
| Formule de paramètres à 0,11 % près | **mesuré** — 4 198 912 contre 4 194 304 |
| Le noyau d'attention change le coût des activations d'un facteur 2,2 | **mesuré** — ablation, même modèle |
| La formule du playbook surestime à longue séquence | **mesuré** ici, sur ce modèle et ce noyau |
| Recompute : mémoire contre temps | **mesuré** ici (petit modèle, CPU) |
| ZeRO-1/2/3 réduisent l'état par rang | **dérivé** du ledger mesuré, avec contrôle de conservation |
| Anneau all-reduce à $2(ND-1)M$, optimal en bande passante | **mesuré** sur tenseurs réels, ND ≤ 8 |
| Les 512 GPU tiennent un 70B à 46 % de MFU | **rapporté** — aucun cluster ici |
| Le parallélisme de tenseur découpe les matrices par rang | **rapporté** — non exécutable sans collectif réel |
| Le recouvrement communication/calcul masque l'all-reduce | **rapporté** — pas de vrai `NCCL` sur un rang |
| Les courbes de scalabilité (jusqu'à 512 GPU) | **hors de portée** — un seul GPU consumer, pas de cluster |

Deux lignes méritent d'être soulignées. La dernière dit qu'aucune conclusion de ce carnet ne porte sur
**quelle** stratégie gagne à grande échelle : cela demande le banc d'essai du playbook. Et l'avant-dernière
explique pourquoi la partie 6 simule l'anneau en Python plutôt que d'appeler `torch.distributed` : sur un
seul rang, un collectif réel ne communique avec personne et ne mesurerait rien.

## Vérifiez votre compréhension

Les trois exercices ci-dessous s'appuient sur les instruments déjà définis plus haut — aucun ne demande
d'écrire un nouveau modèle. Chacun se termine par un `print` que vous pouvez confronter au résultat
attendu, commenté dans l'énoncé.

### Exercice 1 — Combien de cartes pour un 70B ?

Avec le ledger de la partie 2 (20 octets par paramètre) et un budget de 80 Go par carte dont on réserve
20 % pour les activations, quel est le plus petit nombre de rangs $ND$ (une puissance de deux) qui fait
tenir un modèle de 70 milliards de paramètres sous ZeRO-3 ?

Indice : la partie 5 donne `octets_par_param("ZeRO-3", nd)`, et la contrainte s'écrit
$20/ND \times 70 \cdot 10^9 \le 0{,}8 \times 80 \cdot 10^9$.

In [8]:
def rangs_necessaires(milliards, gpu_gio=80, reserve=0.2):
    """Plus petite puissance de deux telle que l'etat ZeRO-3 tienne sur une carte.

    TODO etudiant : partir de octets_par_param("ZeRO-3", nd) * milliards * 1e9,
    convertir en Gio, comparer au budget utile (gpu_gio * (1 - reserve)),
    et doubler nd jusqu'a ce que ca tienne.
    """
    # result = None  # TODO etudiant
    print("Exercice a completer")
    return None


resultat = rangs_necessaires(70)
print("rangs necessaires pour un 70B :", resultat)
print("attendu : 32  (soit 20/32 x 70 Go = 43,75 Go d'etat par carte, sous 64 Go utiles)")

Exercice a completer
rangs necessaires pour un 70B : None
attendu : 32  (soit 20/32 x 70 Go = 43,75 Go d'etat par carte, sous 64 Go utiles)


### Exercice 2 — La recomputation paie-t-elle sur une longue séquence ?

Mesurez les activations de `modele_bf16` à **1 024 jetons**, avec et sans `checkpoint`, et calculez le
facteur d'économie mémoire. Comparez-le au facteur obtenu à 512 jetons dans la partie 4.

Question à trancher par la mesure : l'économie est-elle **constante** en fonction de la séquence, ou
varie-t-elle ? Si elle varie, dans quel sens, et pourquoi ?

In [9]:
def economie_recomputation(seq, lot=1):
    """Facteur d'economie memoire apporte par le checkpoint a cette longueur de sequence.

    TODO etudiant : appeler mesurer_activations deux fois (avec et sans checkpoint),
    verifier que la mesure avec checkpoint est bien inferieure, et rendre le rapport.
    """
    # facteur = None  # TODO etudiant
    print("Exercice a completer", seq)
    return None


for seq in (512, 1024):
    print(f"seq={seq:>5} : facteur = {economie_recomputation(seq)}")

Exercice a completer 512
seq=  512 : facteur = None
Exercice a completer 1024
seq= 1024 : facteur = None


### Exercice 3 — L'anneau reste-t-il optimal à 16 rangs ?

Faites tourner `anneau_all_reduce(16, 4096)` et vérifiez que le volume échangé vaut toujours
$2(ND-1)M$ au ratio $1{,}0000$, et que le résultat est juste sur les seize rangs.

Puis répondez par le calcul : quel est le volume **par rang**, rapporté à la taille du tampon, à
$ND = 16$ ? Que devient ce rapport quand $ND \to \infty$, et pourquoi cela signifie-t-il qu'**ajouter des
cartes ne coûte pas proportionnellement plus cher** en communication ?

In [10]:
def verifier_anneau(nd, taille):
    """Verifie volume et justesse de l'anneau, et rend le volume par rang rapporte a M.

    TODO etudiant : appeler anneau_all_reduce, comparer le volume a 2*(nd-1)*taille*8,
    verifier chaque rang contre la somme attendue, puis calculer
    (volume / nd) / (taille * 8).
    """
    # volume_par_rang = None  # TODO etudiant
    print("Exercice a completer", nd)
    return None


print("ND=16 :", verifier_anneau(16, 4096))
print("attendu : ratio 1.0000, juste=True, volume par rang = 1.875 M")

Exercice a completer 16
ND=16 : None
attendu : ratio 1.0000, juste=True, volume par rang = 1.875 M


## À retenir

1. **L'état d'entraînement coûte 20 octets par paramètre**, pas 4. Les poids ne sont qu'un poste sur six ;
   l'optimiseur et l'accumulation fp32 en portent les trois quarts. Un modèle 7B, c'est 130 Gio avant la
   première activation.
2. **Les activations ne sont pas une propriété du modèle mais du couple (modèle, noyau d'attention).**
   Deux implémentations qui calculent exactement la même chose diffèrent d'un facteur 2,2 en mémoire.
   Toute formule de dimensionnement doit nommer l'implémentation qu'elle décrit.
3. **Une formule analytique se vérifie en comptant les octets réels**, avec un instrument qui ne double pas
   les vues (`numel × element_size`, jamais `untyped_storage().nbytes()`). Le piège est silencieux :
   il rend un chiffre plausible et faux.
4. **La recomputation est un taux de change, pas une magie.** Elle divise la mémoire des activations par
   un facteur déterministe, et multiplie le temps par un facteur qui, lui, dépend de la plateforme — sur
   ce poste CPU le surcoût est de 3 %, là où le playbook annonce +30 à 40 % sur GPU.
5. **ZeRO partage ce qui est identique, et rien d'autre.** ZeRO-1 ne descend pas sous 12 o/param —
   c'est une asymptote, pas un plateau ($16{,}00$ à 2 rangs, $13{,}00$ à 8) : le poste non partitionné
   fixe le plancher ; seul ZeRO-3 descend en $1/ND$. Les rendements décroissent, et le coût de
   communication, lui, croît.
6. **L'anneau all-reduce est optimal en bande passante** : $\frac{2(ND-1)}{ND}M$ par rang, soit moins du
   double du tampon quel que soit le nombre de cartes. C'est ce qui rend ZeRO praticable au-delà de deux
   cartes.
7. **Un carnet sur un article de 512 GPU doit dire lequel des deux parle.** La table de la partie 7 n'est
   pas une précaution de style : c'est ce qui sépare une mesure locale d'une croyance recopiée.

**Pour aller plus loin.** L'Ultra-Scale Playbook de Hugging Face est en libre accès :
<https://huggingface.co/spaces/nanotron/ultrascale-playbook>. Il contient le banc d'essai, les courbes de
scalabilité jusqu'à 512 GPU, le parallélisme de tenseur, de pipeline et de contexte, et les stratégies de
recouvrement que ce carnet ne fait que nommer. Ce carnet est la partie vérifiable sans cluster ; le reste
demande le cluster.